<a href="https://colab.research.google.com/github/nafeessiddique-dev/XAIApproxRL/blob/main/Final_IG_VGG16_multiplier_select_homogeneous.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision torchaudio --no-cache-dir

In [ ]:
!pip install torch-xla
!pip install opencv-python
!pip install numpy torchao
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.8 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# updated focus

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import os
from torchvision import models, datasets, transforms
from PIL import Image
import shutil
import math
from torch.utils.cpp_extension import load
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pickle

# =============================================================================
# MAC CONFIGURATIONS - ALL EVOAPPROX MULTIPLIERS
# =============================================================================
MAC_CONFIGURATIONS = {
    'mul8s_1KV6_add16se_2TN': {'multiplier': 'mul8s_1KV6', 'adder': 'add16se_2TN', 'mul_power_mW': 0.425, 'mul_delay_ns': 1.48, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.00,  'mre': 0.00,    'level': 0},
    'mul8s_1KV8_add16se_2TN': {'multiplier': 'mul8s_1KV8', 'adder': 'add16se_2TN', 'mul_power_mW': 0.422, 'mul_delay_ns': 1.48, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.0018,'mre': 0.28,    'level': 1},
    'mul8s_1KV9_add16se_2TN': {'multiplier': 'mul8s_1KV9', 'adder': 'add16se_2TN', 'mul_power_mW': 0.410, 'mul_delay_ns': 1.47, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.0064,'mre': 0.90,    'level': 2},
    'mul8s_1KVA_add16se_2TN': {'multiplier': 'mul8s_1KVA', 'adder': 'add16se_2TN', 'mul_power_mW': 0.391, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.019, 'mre': 2.53,    'level': 3},
    'mul8s_1KVP_add16se_2TN': {'multiplier': 'mul8s_1KVP', 'adder': 'add16se_2TN', 'mul_power_mW': 0.363, 'mul_delay_ns': 1.37, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.051, 'mre': 2.73,    'level': 4},
    'mul8s_1KVQ_add16se_2TN': {'multiplier': 'mul8s_1KVQ', 'adder': 'add16se_2TN', 'mul_power_mW': 0.351, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.056, 'mre': 3.64,    'level': 5},
    'mul8s_1KX5_add16se_2TN': {'multiplier': 'mul8s_1KX5', 'adder': 'add16se_2TN', 'mul_power_mW': 0.289, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.15,  'mre': 8.93,    'level': 6},
    'mul8s_1L2J_add16se_2TN': {'multiplier': 'mul8s_1L2J', 'adder': 'add16se_2TN', 'mul_power_mW': 0.301, 'mul_delay_ns': 1.36, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.081, 'mre': 4.41,    'level': 7},
    'mul8s_1L2L_add16se_2TN': {'multiplier': 'mul8s_1L2L', 'adder': 'add16se_2TN', 'mul_power_mW': 0.200, 'mul_delay_ns': 1.14, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.23,  'mre': 12.26,   'level': 8},
    'mul8s_1L2N_add16se_2TN': {'multiplier': 'mul8s_1L2N', 'adder': 'add16se_2TN', 'mul_power_mW': 0.126, 'mul_delay_ns': 0.94, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.52,  'mre': 27.44,   'level': 9},
    'mul8s_1L12_add16se_2TN': {'multiplier': 'mul8s_1L12', 'adder': 'add16se_2TN', 'mul_power_mW': 0.052, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 3.08,  'mre': 135.77,  'level': 10}
}

# =============================================================================
# HARDWARE CONFIG
# =============================================================================
HARDWARE_CONFIG = {
    'clock_period_ns':        3.0,
    'pe_array_size':          65536,
    'load_weights_cycles':    100,
    'sram_energy_pJ_per_access': 1.92,
}

# =============================================================================
# REWARD WEIGHTS
# =============================================================================
REWARD_WEIGHTS = {
    'energy_efficiency':      0.20,
    'spearman_correlation':   0.50,
    'top_pixel_concentration':0.30
}
# Quality gate: energy reward is zeroed if per-image Spearman drops below
# this fraction of the cluster's best (least-approximate) Spearman score.
# Prevents aggressive multipliers winning on energy when quality degrades.
SPEARMAN_GATE_FRACTION = 0.97   # must retain 97% of best-in-cluster Spearman

# =============================================================================
# CLUSTERING CONFIG
# =============================================================================
CLUSTERING_CONFIG = {
    'n_clusters':    5,     # K for K-Means
    'pca_components':128,   # reduce VGG16 penultimate 4096-D → 128-D
    'apply_pca':     True,  # VGG16 penultimate is 4096-D → apply PCA
    'random_state':  42
}

# =============================================================================
# VGG16 CLASSIFIER — 37 CLASSES  (unchanged)
# =============================================================================
class VGG16Classifier(nn.Module):
    def __init__(self, num_classes=37):
        super(VGG16Classifier, self).__init__()
        self.model = models.vgg16(weights=None)
        in_features = self.model.classifier[6].in_features
        self.model.classifier[6] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return F.log_softmax(self.model(x), dim=1)


# =============================================================================
# PENULTIMATE FEATURE EXTRACTOR
# VGG16 classifier layout:
#   [0] Linear(25088→4096)  [1] ReLU  [2] Dropout
#   [3] Linear(4096→4096)   [4] ReLU  [5] Dropout  [6] Linear(4096→37)
# Penultimate = output of classifier[4] → 4096-D vector
# =============================================================================
class PenultimateExtractor(nn.Module):
    def __init__(self, full_model):
        super(PenultimateExtractor, self).__init__()
        self.features   = full_model.model.features
        self.avgpool    = full_model.model.avgpool
        # up to and including index 4 (second ReLU = penultimate layer)
        self.classifier = nn.Sequential(
            *list(full_model.model.classifier.children())[:5]
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x   # (batch, 4096)


# =============================================================================
# VISUAL FEATURE EXTRACTION
# 5 features: edge density, center edge ratio, texture energy, entropy, contrast
# =============================================================================
def extract_visual_features(img_array: np.ndarray) -> np.ndarray:
    """
    img_array: (224,224,3) float32 in [0,1]
    Returns:   (5,) float32 numpy array
    """
    gray   = (0.299 * img_array[:,:,0] +
               0.587 * img_array[:,:,1] +
               0.114 * img_array[:,:,2])
    gray_u8 = np.uint8(gray * 255)

    # 1. Edge density
    edges        = cv2.Canny(gray_u8, 50, 150)
    edge_density = float(np.sum(edges > 0)) / edges.size

    # 2. Center edge ratio
    H, W         = edges.shape
    hm, wm       = H // 4, W // 4
    center_edges = edges[hm:H-hm, wm:W-wm]
    full_sum     = float(np.sum(edges)) + 1e-8
    center_edge_ratio = float(np.sum(center_edges)) / full_sum

    # 3. Texture energy
    lap           = cv2.Laplacian(gray_u8, cv2.CV_64F)
    texture_energy= float(np.mean(lap ** 2))

    # 4. Image entropy
    hist, _  = np.histogram(gray_u8, bins=256, range=(0, 256))
    prob     = hist / (hist.sum() + 1e-8)
    entropy  = float(-np.sum(prob * np.log2(prob + 1e-8)))

    # 5. Contrast (RMS)
    contrast = float(np.std(gray))

    return np.array([edge_density, center_edge_ratio,
                     texture_energy, entropy, contrast], dtype=np.float32)


# =============================================================================
# METRICS  (unchanged)
# =============================================================================
def calculate_spearman_correlation(map1, map2):
    f1, f2 = map1.flatten(), map2.flatten()
    if np.std(f1) == 0 or np.std(f2) == 0:
        return 0.0
    corr, _ = spearmanr(f1, f2)
    return 0.0 if np.isnan(corr) else float(corr)


def calculate_top_pixel_concentration(gradient_map):
    """
    Multi-scale focus concentration (from notebook formula):
        f1 = sum of top  5% values / sum of all values
        f2 = sum of top 10% values / sum of all values
        f3 = sum of top 15% values / sum of all values
        f4 = sum of top 20% values / sum of all values
        f  = (f1 + f2 + f3 + f4) / 4
    """
    abs_g    = np.abs(gradient_map.flatten())
    sorted_g = np.sort(abs_g)[::-1]
    total    = np.sum(sorted_g)
    if total == 0:
        return 0.0
    n        = len(sorted_g)
    f1 = float(np.sum(sorted_g[:max(1, int(n * 0.05))]) / total)
    f2 = float(np.sum(sorted_g[:max(1, int(n * 0.10))]) / total)
    f3 = float(np.sum(sorted_g[:max(1, int(n * 0.15))]) / total)
    f4 = float(np.sum(sorted_g[:max(1, int(n * 0.20))]) / total)
    return (f1 + f2 + f3 + f4) / 4.0


def create_high_res_red_gradient_map(gradient_map_normalized, original_size,
                                     apply_red=True):
    resized = cv2.resize(gradient_map_normalized, original_size,
                         interpolation=cv2.INTER_CUBIC)
    g8      = np.uint8(255 * resized)
    if apply_red:
        out = np.zeros((*g8.shape, 3), dtype=np.uint8)
        out[:,:,2] = g8
        return out
    return cv2.cvtColor(g8, cv2.COLOR_GRAY2BGR)


# =============================================================================
# ROHNAS ENERGY MODEL  (unchanged)
# =============================================================================
class RoHNASEnergyModel:
    def __init__(self, mac_config_name):
        cfg = MAC_CONFIGURATIONS[mac_config_name]
        self.config_name         = mac_config_name
        self.approximation_level = cfg['level']
        self.mac_energy_pJ       = cfg['mul_power_mW'] + cfg['add_power_mW']
        self.T                   = HARDWARE_CONFIG['clock_period_ns']
        self.pe_array_size       = HARDWARE_CONFIG['pe_array_size']
        self.E_memory            = HARDWARE_CONFIG['sram_energy_pJ_per_access']

    def calculate_layer_energy_rohnas(self, layer_desc):
        wl   = layer_desc.get_weights_count()
        sl   = layer_desc.get_sum_count()
        fl   = layer_desc.get_feature_maps_count()
        wPE  = math.ceil(wl / (self.pe_array_size * min(self.pe_array_size, sl)))
        m_acc= 65536 if fl == 1 else self.pe_array_size * max(sl - 15, 1)
        cl   = wl * wPE + fl
        return cl * self.T, (m_acc / 128.0) * self.E_memory + cl * self.mac_energy_pJ

    def calculate_ig_energy(self, n_steps, layer_descriptors):
        total = sum(self.calculate_layer_energy_rohnas(ld)[1]
                    for ld in layer_descriptors)
        return {'ig_energy_mJ': total * n_steps * 2 / 1e9,
                'config_name':  self.config_name}


# =============================================================================
# LAYER DESCRIPTOR  (unchanged)
# =============================================================================
class LayerDescriptor:
    def __init__(self, layer_type, name, input_shape, output_shape,
                 kernel_size=1, stride=1, params=None):
        self.layer_type   = layer_type
        self.name         = name
        self.input_shape  = input_shape
        self.output_shape = output_shape
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.params       = params or {}
        if layer_type == 'conv':
            self.chin, self.hin, self.win    = input_shape
            self.chout, self.hout, self.wout = output_shape
            self.nin, self.nout = self.hin, self.hout
        elif layer_type == 'fc':
            self.chin  = input_shape[0]  if isinstance(input_shape,  tuple) else input_shape
            self.chout = output_shape[0] if isinstance(output_shape, tuple) else output_shape
            self.nin = self.nout = 1

    def get_weights_count(self):
        if self.layer_type == 'conv': return self.kernel_size**2 * self.chin * self.chout
        if self.layer_type == 'fc':  return self.chin * self.chout
        return 0

    def get_sum_count(self):
        if self.layer_type == 'conv': return self.kernel_size**2 * self.chin
        if self.layer_type == 'fc':  return self.chin
        return 0

    def get_feature_maps_count(self):
        if self.layer_type == 'conv': return self.nout * self.nout
        if self.layer_type == 'fc':  return 1
        return 0


class LayerProfiler:
    def __init__(self, model, input_shape=(1, 3, 224, 224)):
        self.model, self.input_shape = model, input_shape
        self.layer_descriptors, self.hooks = [], []

    def _register_hooks(self):
        def hook_fn(name, _):
            def hook(module, inp, out):
                inp   = inp[0] if isinstance(inp, tuple) else inp
                in_s  = tuple(inp.shape[1:])
                out_s = tuple(out.shape[1:])
                if isinstance(module, nn.Conv2d):
                    ks = module.kernel_size[0] if isinstance(module.kernel_size, tuple) else module.kernel_size
                    st = module.stride[0]      if isinstance(module.stride,      tuple) else module.stride
                    self.layer_descriptors.append(
                        LayerDescriptor('conv', name, in_s, out_s, ks, st))
                elif isinstance(module, nn.Linear):
                    chin  = in_s[0] if len(in_s) == 1 else int(np.prod(in_s))
                    self.layer_descriptors.append(
                        LayerDescriptor('fc', name, (chin,), (out_s[0],)))
            return hook
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                self.hooks.append(module.register_forward_hook(hook_fn(name, '')))

    def profile(self):
        self._register_hooks()
        self.model.eval()
        with torch.no_grad():
            self.model(torch.randn(*self.input_shape))
        for h in self.hooks: h.remove()
        return self.layer_descriptors


# =============================================================================
# APPROXIMATE MULTIPLIER HELPERS  (unchanged)
# =============================================================================
def load_approximate_multiplier_kernel(axx_mult):
    return load(
        name=f'PyInit_approx_mult_{axx_mult}',
        sources=["/content/adapt/cpu-kernels/axx_linear.cpp"],
        extra_cflags=[f'-DAXX_MULT={axx_mult} -march=native -fopenmp -O3'],
        extra_include_paths=["/content/adapt/cpu-kernels"],
        extra_ldflags=['-lgomp'],
        verbose=False
    )


def approx_matmul_submatrix_pt(A_sub, B_sub, axx_kernel):
    r, c = A_sub.shape; cb = B_sub.shape[1]
    if r == 0 or c == 0 or cb == 0:
        return torch.zeros((r, cb), dtype=torch.float32)
    s  = 127
    qA = torch.clamp(A_sub * s, -128, 127).to(torch.int8)
    qB = torch.clamp(B_sub * s, -128, 127).to(torch.int8)
    return axx_kernel.forward(qA, qB.t().contiguous()).to(torch.float32) / (s * s)


def solve_vandermonde_pt(xs, ys, axx_kernel):
    n  = xs.shape[0]
    V  = torch.pow(xs.unsqueeze(1).repeat(1, n), torch.arange(n).float())
    Vi = torch.linalg.inv(V)
    return approx_matmul_submatrix_pt(Vi, ys.unsqueeze(1), axx_kernel).squeeze()


def polynomial_interpolation_pt(coefficients, alphas):
    cr     = coefficients.flip(dims=[0])
    result = torch.zeros_like(alphas)
    for i, alpha in enumerate(alphas):
        y = cr[0]
        for j in range(1, len(cr)): y = y * alpha + cr[j]
        result[i] = y
    return result


# =============================================================================
# INTEGRATED GRADIENTS  (unchanged)
# =============================================================================
def compute_integrated_gradients(model, image_input, baseline, steps,
                                 axx_mult, device):
    alphas     = np.linspace(0.0, 1.0, steps + 1)
    img_nchw   = image_input.permute(2, 0, 1).unsqueeze(0)
    base_nchw  = baseline.permute(2, 0, 1).unsqueeze(0)
    axx_kernel = load_approximate_multiplier_kernel(axx_mult)

    ys = []
    for a in alphas:
        interp = base_nchw + a * (img_nchw - base_nchw)
        with torch.no_grad():
            pred = model(interp)
            ys.append(torch.exp(pred)[0, 0].item())
    ys = np.array(ys)

    alphas_pt = torch.tensor(alphas, dtype=torch.float32, device=device)
    ys_pt     = torch.tensor(ys,     dtype=torch.float32, device=device)
    coeffs    = solve_vandermonde_pt(alphas_pt, ys_pt, axx_kernel)
    interp_v  = polynomial_interpolation_pt(coeffs, alphas_pt)

    ig = torch.zeros_like(image_input, device=device)
    for i in range(steps):
        for a_val in [interp_v[i], interp_v[i + 1]]:
            pt = (base_nchw + a_val * (img_nchw - base_nchw)).detach().clone()
            pt.requires_grad_(True)
            with torch.set_grad_enabled(True):
                pred = model(pt)
                grad = torch.autograd.grad(pred[0, 0], pt)[0].squeeze(0)
            ig += grad.permute(1, 2, 0).detach() * (alphas[i+1] - alphas[i]) / 2.0

    sig  = (image_input - baseline) * ig / steps
    gmap = np.abs(sig.cpu().detach().numpy()).max(axis=-1)
    mx   = gmap.max()
    return gmap / mx if mx > 0 else gmap


# =============================================================================
# =====================  OFFLINE PHASE  =======================================
# =============================================================================
def offline_phase(model, images_data, device, layer_descriptors,
                  all_energy_mJ, E_max, energy_eff_map, output_dir, approx_configs, steps=5):
    """
    OFFLINE PHASE — runs once at design time.

    Steps (strictly following the outline):
      1. Detect TPU cores and multipliers          → printed in main
      2. Collect training images                   → passed in as images_data
      3. Extract penultimate features + PCA
      4. Extract visual features, concatenate
      5. K-Means clustering — store centroids, labels, image→feature map
      6. Compute accurate gradient maps
      7. Collect reward R per image per multiplier
      8. Calculate average J(mp) per cluster
      9. Choose multiplier with highest J(mp)
     10. Finalize and save cluster→multiplier→core mapping
    """
    os.makedirs(output_dir, exist_ok=True)
    # approx_configs passed in from main — contains only the P selected multipliers

    # ------------------------------------------------------------------
    # STEP 3 — Compute accurate logits for all training images
    # ------------------------------------------------------------------
    # Sensitivity signal: Mean Absolute Difference (MAD) between accurate
    # and approximate logits.
    #
    # Why logit-MAD instead of Spearman on gradient maps:
    #   - Gradient map Spearman was 0.999 for KVA→KX5 because spatial
    #     averaging washes out small approximation errors.
    #   - Logits are 37-D vectors with no spatial averaging — even a tiny
    #     approximation error in a Linear layer shifts logit values by a
    #     measurable amount, giving real signal for mild multipliers.
    #   - No hooks, no patching, no IG — just two forward passes per
    #     multiplier per image: one accurate, one with approx Linear layers.
    #
    # Approximate forward pass: patch each nn.Linear's weight matrix
    # through approx_matmul_submatrix_pt, run the full model, restore.
    # Conv layers use standard float ops (approx error is in MAC hardware).
    # ------------------------------------------------------------------
    print("\n" + "="*70)
    print("OFFLINE — STEP 3: Accurate Logits")
    print("="*70)

    # image_feature_map / pca_model / scaler kept for mapping_data compat
    image_feature_map = {i: {'breed': images_data[i].get('breed', '?'),
                              'penult_idx': i}
                         for i in range(len(images_data))}
    pca_model = None
    scaler    = StandardScaler()

    def run_forward(img_arr, axx_mult=None):
        """
        Single forward pass → (37,) log-softmax logit vector.
        axx_mult=None  → accurate (plain PyTorch).
        axx_mult=str   → approximate: Linear layers patched with approx kernel.
        """
        img_t = (torch.from_numpy(img_arr)
                 .permute(2, 0, 1).unsqueeze(0).float())

        if axx_mult is None:
            with torch.no_grad():
                out = model(img_t)
            return out.squeeze(0).cpu().numpy()   # (37,)

        # Patch Linear layers
        axx_kernel    = load_approximate_multiplier_kernel(axx_mult)
        orig_forwards = {}
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                orig_forwards[name] = module.forward
                W_ = module.weight.data
                b_ = module.bias.data if module.bias is not None else None
                def _make(W__, b__, kern__):
                    def _fwd(x):
                        o = approx_matmul_submatrix_pt(x, W__.t(), kern__)
                        return o + b__ if b__ is not None else o
                    return _fwd
                module.forward = _make(W_, b_, axx_kernel)
        with torch.no_grad():
            try:
                out = model(img_t).squeeze(0).cpu().numpy()
            except Exception:
                out = np.zeros(37, dtype=np.float32)
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear) and name in orig_forwards:
                module.forward = orig_forwards[name]
        out = np.where(np.isfinite(out), out, 0.0)
        return out.astype(np.float32)

    accurate_logits = []
    for i, img_data in enumerate(images_data):
        logit = run_forward(img_data['array'], axx_mult=None)
        accurate_logits.append(logit)
        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{len(images_data)}] done")
    accurate_logits = np.array(accurate_logits)   # (N, 37)
    print(f"  Accurate logits shape: {accurate_logits.shape}")

    # ------------------------------------------------------------------
    # STEP 4 — Compute approximate logits + build logit-MAD sensitivity
    #          profile matrix
    # ------------------------------------------------------------------
    # sensitivity_profiles[i, m] = mean |logit_acc[i] - logit_approx[i][m]|
    # Higher MAD = image is more sensitive to multiplier m.
    # Because logits are un-averaged 37-D vectors, even KVA produces a
    # non-zero MAD distinguishable from KVP, KVQ, KX5, L2L.
    # ------------------------------------------------------------------
    print("\nOFFLINE — STEP 4: Approximate Logits + Logit-MAD Sensitivity Matrix")

    N = len(images_data)
    P = len(approx_configs)
    sensitivity_profiles = np.zeros((N, P), dtype=np.float32)

    for ci, config_name in enumerate(approx_configs):
        mult = MAC_CONFIGURATIONS[config_name]['multiplier']
        print(f"  [{ci+1}/{P}] {config_name}  (mult={mult})")
        for idx in range(N):
            apx_logit = run_forward(images_data[idx]['array'], axx_mult=mult)
            mad = float(np.mean(np.abs(accurate_logits[idx] - apx_logit)))
            sensitivity_profiles[idx, ci] = mad

    print(f"  Sensitivity profile shape: {sensitivity_profiles.shape}")
    print(f"  MAD range per multiplier:")
    for ci, cfg in enumerate(approx_configs):
        col = sensitivity_profiles[:, ci]
        print(f"    {cfg.split('_')[1]:<8}  "
              f"min={col.min():.4f}  max={col.max():.4f}  "
              f"mean={col.mean():.4f}")

    # ------------------------------------------------------------------
    # STEP 5 — K-Means clustering on logit-MAD sensitivity profiles
    # ------------------------------------------------------------------
    # Each image is now a (P,) vector of MAD values — one per multiplier.
    # Images with high MAD at aggressive multipliers cluster together as
    # "sensitive"; images with low MAD across all levels cluster as
    # "tolerant". K-Means on this space produces genuinely different
    # sensitivity groups.
    # ------------------------------------------------------------------
    print("\nOFFLINE — STEP 5: K-Means on Logit-MAD Profiles")

    K             = CLUSTERING_CONFIG['n_clusters']
    sens_scaler   = StandardScaler()
    sens_scaled   = sens_scaler.fit_transform(sensitivity_profiles)
    kmeans_sens   = KMeans(n_clusters=K,
                           random_state=CLUSTERING_CONFIG['random_state'],
                           n_init=20)
    cluster_labels        = kmeans_sens.fit_predict(sens_scaled)
    sensitivity_centroids = kmeans_sens.cluster_centers_
    sensitivity_scaler    = sens_scaler
    centroids             = sensitivity_centroids

    print(f"  K = {K} logit-MAD clusters formed")
    for k in range(K):
        idxs_k  = [i for i in range(N) if cluster_labels[i] == k]
        members = [images_data[i].get('breed', str(i)) for i in idxs_k]
        mean_mad_l2l = float(np.mean(
            sensitivity_profiles[idxs_k, -1])) if idxs_k else 0.0
        print(f"  Cluster {k} ({len(members)} images, "
              f"mean_L2L_MAD={mean_mad_l2l:.4f}): "
              f"{members[:6]}{'...' if len(members) > 6 else ''}")

    # ------------------------------------------------------------------
    # STEP 6 — Compute accurate gradient maps
    # ------------------------------------------------------------------
    print("\nOFFLINE — STEP 6: Accurate Gradient Maps")

    accurate_gmaps = []
    for i, img_data in enumerate(images_data):
        img_t    = torch.from_numpy(img_data['array']).float().to(device)
        baseline = torch.zeros_like(img_t)
        gmap     = compute_integrated_gradients(
            model, img_t, baseline, steps, axx_mult='mul8s_1KV6', device=device)
        accurate_gmaps.append(gmap)
        print(f"  [{i+1}/{len(images_data)}] {img_data.get('breed','?')}")
    print(f"  ✓ {len(accurate_gmaps)} accurate gradient maps computed")

    # ------------------------------------------------------------------
    # STEPS 7 & 8 — Collect reward R per image, calculate J(mp) per cluster
    # ------------------------------------------------------------------
    print("\nOFFLINE — STEPS 7 & 8: Reward Collection + J(mp) Calculation")

    # ── Energy summary table ──────────────────────────────────────────
    print("\n  ENERGY TABLE (approx range KVA → L2L)")
    print(f"  {'Multiplier':<35} {'Level':<8} {'Energy(mJ)':<14} {'Energy_eff':<12} {'MAE':<8} {'MRE%'}")
    print("  " + "-"*85)
    for cfg in approx_configs:
        c   = MAC_CONFIGURATIONS[cfg]
        emJ = all_energy_mJ[cfg]
        eff = energy_eff_map[cfg]
        print(f"  {cfg:<35} {c['level']:<8} {emJ:<14.4f} {eff:<12.4f} {c['mae']:<8.4f} {c['mre']:.2f}")

    # ── Pre-compute ALL approx gradient maps once ─────────────────────
    # approx_gmaps_cache[img_idx][config_name] = normalised gradient map
    # Used for reward logging and saving visualisations.
    print("\n  Pre-computing approximate gradient maps (for reward logging)...")
    approx_gmaps_cache = {i: {} for i in range(len(images_data))}
    for ci, config_name in enumerate(approx_configs):
        mult = MAC_CONFIGURATIONS[config_name]['multiplier']
        print(f"    [{ci+1}/{len(approx_configs)}] {config_name}")
        for idx in range(len(images_data)):
            img_t    = torch.from_numpy(images_data[idx]['array']).float().to(device)
            baseline = torch.zeros_like(img_t)
            approx_gmaps_cache[idx][config_name] = compute_integrated_gradients(
                model, img_t, baseline, steps, axx_mult=mult, device=device)
    print("  ✓ All approximate gradient maps cached")


    # reward_table[k][config] accumulates per-image rewards (for logging only)
    reward_table = {k: {cfg: [] for cfg in approx_configs} for k in range(K)}

    # Store sensitivity clustering info for online routing
    mapping_data_extra = {
        'sensitivity_centroids': sensitivity_centroids,
        'sensitivity_scaler':    sensitivity_scaler,
        'approx_configs':        approx_configs,
    }

    # ------------------------------------------------------------------
    # STEP 9 — Activation-drift ranking → guaranteed variety assignment
    # ------------------------------------------------------------------
    # Sensitivity signal: per-cluster mean activation drift at each
    # multiplier level (higher drift = more sensitive to approximation).
    # Because drift is measured per layer, not just at the output,
    # even mild multipliers (KVA→KX5) produce measurably different drift
    # values across clusters — fixing the 0.999 Spearman plateau.
    #
    # Assignment logic (same guaranteed-variety approach as before, but
    # now using a real signal):
    #   1. For each cluster, compute mean activation drift at L2L
    #      (most aggressive mult) — higher drift = more sensitive.
    #   2. Rank clusters by L2L drift: rank 0 = most sensitive (highest
    #      drift), rank K-1 = most tolerant (lowest drift).
    #   3. Map rank → multiplier band: rank 0 → KVA, rank K-1 → L2L.
    #   4. Apply aggressiveness shift from energy weight.
    #
    # Effect of REWARD_WEIGHTS['energy_efficiency']:
    #   w_energy=0.00 → shift=-1 (one step more conservative)
    #   w_energy=0.33 → shift= 0 (neutral)
    #   w_energy=0.67 → shift=+1 (one step more aggressive)
    #   w_energy=1.00 → shift=+2 (two steps more aggressive)
    # Changing w_energy now has a guaranteed visible effect.
    # ------------------------------------------------------------------
    print("\nOFFLINE — STEP 9: Activation-Drift Ranking → Variety Assignment")

    # ── Step 1: Compute per-cluster mean logit-MAD at each multiplier ──
    # cluster_mad[k][ci] = mean logit-MAD for cluster k at multiplier ci
    cluster_mad = {}
    print(f"  Per-cluster mean logit-MAD:")
    print(f"  {'Cluster':<10}", end="")
    for cfg in approx_configs:
        print(f"  {cfg.split('_')[1]:<10}", end="")
    print()
    print("  " + "-"*70)

    for k in range(K):
        idxs = [int(i) for i, lbl in enumerate(cluster_labels) if lbl == k]
        mads = []
        for ci, cfg in enumerate(approx_configs):
            mad = float(np.mean(sensitivity_profiles[idxs, ci])) if idxs else 0.0
            mads.append(mad)
        cluster_mad[k] = mads
        row = f"  {k:<10}"
        for d in mads:
            row += f"  {d:<10.4f}"
        print(row)

    # ── Step 2: Rank clusters by L2L MAD (most sensitive first) ───────
    l2l_mad_per_cluster = {k: cluster_mad[k][-1] for k in range(K)}
    sorted_clusters = sorted(range(K),
                             key=lambda k: l2l_mad_per_cluster[k],
                             reverse=True)   # highest MAD = most sensitive

    print(f"\n  Clusters ranked by L2L MAD (most sensitive first):")
    for rank, k in enumerate(sorted_clusters):
        print(f"    Rank {rank}: Cluster {k}  "
              f"(L2L_MAD={l2l_mad_per_cluster[k]:.4f})")

    # ── Step 3: Map rank → multiplier band ───────────────────────────
    P_mults = len(approx_configs)
    cluster_to_band = {}
    for rank, k in enumerate(sorted_clusters):
        band = int(rank * P_mults / K)
        band = min(band, P_mults - 1)
        cluster_to_band[k] = band

    # ── Step 4: Apply energy-weight aggressiveness shift ─────────────
    w_e   = float(REWARD_WEIGHTS['energy_efficiency'])
    shift = round((w_e - 0.33) * 3)
    print(f"\n  Energy weight={w_e:.2f} → aggressiveness shift={shift:+d}")
    print(f"  (neutral at w_energy=0.33; ±1 per ±0.33)")

    cluster_to_multiplier = {}
    print(f"\n  {'Cluster':<10} {'L2L_MAD':<12} {'Band':<6} "
          f"{'Shifted':<10} {'Multiplier':<35} {'Level'}")
    print("  " + "-"*85)
    for k in range(K):
        base_band    = cluster_to_band[k]
        shifted_band = min(max(base_band + shift, 0), P_mults - 1)
        chosen_cfg   = approx_configs[shifted_band]
        cluster_to_multiplier[k] = chosen_cfg
        c = MAC_CONFIGURATIONS[chosen_cfg]
        print(f"  {k:<10} {l2l_mad_per_cluster[k]:<12.4f} {base_band:<6} "
              f"{shifted_band:<10} {chosen_cfg:<35} Level {c['level']}")

    # ------------------------------------------------------------------
    # STEP 8b — Save gradient maps for representative images
    # ------------------------------------------------------------------
    # For each of the 4 target breeds, find the first matching image,
    # compute accurate + all approximate gradient maps, and save to disk.
    # Target breeds: Saint Bernard, Persian, Russian Blue, Siamese.
    # ------------------------------------------------------------------
    print("\nOFFLINE — STEP 8b: Save Gradient Maps for Representative Images")

    TARGET_BREEDS = ['Saint Bernard', 'Persian', 'Russian Blue', 'Siamese',
                     'Bombay', 'Samoyed']
    gmap_output_dir = os.path.join(output_dir, 'representative_gradient_maps')
    os.makedirs(gmap_output_dir, exist_ok=True)

    for target_breed in TARGET_BREEDS:
        # Find first image matching this breed
        match_idx = next(
            (i for i, d in enumerate(images_data)
             if d.get('breed', '').lower() == target_breed.lower()),
            None)
        if match_idx is None:
            print(f"  ⚠ '{target_breed}' not found in training images — skipping")
            continue

        img_data  = images_data[match_idx]
        breed_tag = target_breed.replace(' ', '_')
        orig_sz   = img_data.get('original_size', (224, 224))
        img_path  = img_data.get('path', '')
        cluster_k = int(cluster_labels[match_idx])

        print(f"  {target_breed}  (img{match_idx}, cluster={cluster_k})")

        breed_dir = os.path.join(gmap_output_dir, breed_tag)
        os.makedirs(breed_dir, exist_ok=True)

        # Save original image
        if os.path.exists(str(img_path)):
            Image.open(img_path).convert('RGB').save(
                os.path.join(breed_dir, f'{breed_tag}_original.png'))

        # Accurate gradient map
        img_t    = torch.from_numpy(img_data['array']).float().to(device)
        baseline = torch.zeros_like(img_t)
        acc_gmap = compute_integrated_gradients(
            model, img_t, baseline, steps, axx_mult='mul8s_1KV6', device=device)
        cv2.imwrite(os.path.join(breed_dir, f'{breed_tag}_ACCURATE_gray.png'),
                    create_high_res_red_gradient_map(acc_gmap, orig_sz, apply_red=False))
        cv2.imwrite(os.path.join(breed_dir, f'{breed_tag}_ACCURATE_red.png'),
                    create_high_res_red_gradient_map(acc_gmap, orig_sz, apply_red=True))

        # Approximate gradient map for the multiplier assigned to this cluster
        assigned_cfg  = cluster_to_multiplier.get(cluster_k, approx_configs[0])
        assigned_mult = MAC_CONFIGURATIONS[assigned_cfg]['multiplier']
        apx_gmap = compute_integrated_gradients(
            model, img_t, baseline, steps, axx_mult=assigned_mult, device=device)
        cv2.imwrite(os.path.join(breed_dir,
                    f'{breed_tag}_{assigned_mult}_gray.png'),
                    create_high_res_red_gradient_map(apx_gmap, orig_sz, apply_red=False))
        cv2.imwrite(os.path.join(breed_dir,
                    f'{breed_tag}_{assigned_mult}_red.png'),
                    create_high_res_red_gradient_map(apx_gmap, orig_sz, apply_red=True))

        # Also save one map per multiplier for comparison
        for cfg in approx_configs:
            mult = MAC_CONFIGURATIONS[cfg]['multiplier']
            gmap = compute_integrated_gradients(
                model, img_t, baseline, steps, axx_mult=mult, device=device)
            cv2.imwrite(os.path.join(breed_dir, f'{breed_tag}_{mult}_gray.png'),
                        create_high_res_red_gradient_map(gmap, orig_sz, apply_red=False))
            cv2.imwrite(os.path.join(breed_dir, f'{breed_tag}_{mult}_red.png'),
                        create_high_res_red_gradient_map(gmap, orig_sz, apply_red=True))

        spearman = calculate_spearman_correlation(acc_gmap, apx_gmap)
        print(f"    Assigned: {assigned_cfg}  Spearman(acc vs apx)={spearman:.4f}")
        print(f"    Saved → {breed_dir}")

    print(f"  ✓ Gradient maps saved → {gmap_output_dir}")

    # ------------------------------------------------------------------
    # STEP 10 — Finalize multiplier assignment per core
    # ------------------------------------------------------------------
    print("\nOFFLINE — STEP 10: Final TPU Core Assignment")
    print("="*70)
    print(f"  {'Cluster':<10} {'Multiplier':<35} {'TPU Core'}")
    print("  " + "-"*55)
    for k, cfg in cluster_to_multiplier.items():
        print(f"  {k:<10} {cfg:<35} Core {k + 1}")

    # Save everything the online phase needs.
    # Online routing now uses sensitivity-based centroids:
    # new image → compute approx gmaps → sensitivity profile → nearest centroid
    mapping_data = {
        'cluster_to_multiplier':  cluster_to_multiplier,
        'centroids':              sensitivity_centroids,   # sensitivity-based
        'sensitivity_scaler':     sensitivity_scaler,
        'approx_configs':         approx_configs,
        # Keep PCA/visual scaler for backward compat (not used in routing now)
        'pca_model':              pca_model,
        'scaler':                 scaler,
        'apply_pca':              CLUSTERING_CONFIG['apply_pca'],
        'cluster_labels':         cluster_labels,
        'image_feature_map':      image_feature_map,
    }
    save_path = os.path.join(output_dir, 'cluster_multiplier_mapping.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(mapping_data, f)
    print(f"\n  ✓ Mapping saved → {save_path}")

    return mapping_data


# =============================================================================
# =====================  ONLINE PHASE  ========================================
# =============================================================================
def online_phase(model, img_data, mapping_data, device,
                 output_dir, steps=20, img_idx=0):
    """
    ONLINE PHASE — runs at runtime for each new image.

    Steps (strictly following the outline):
      1. Extract penultimate layer features (forward pass, same VGG16)
      2. Apply PCA if used in offline phase
      3. Extract visual features, concatenate → enriched feature vector
      4. Find nearest cluster via Euclidean distance to stored centroids
      5. Select corresponding multiplier from saved mapping
      6. Compute gradient map for the new image
    """
    os.makedirs(output_dir, exist_ok=True)

    centroids             = mapping_data['centroids']
    cluster_to_multiplier = mapping_data['cluster_to_multiplier']
    pca_model             = mapping_data.get('pca_model')
    scaler                = mapping_data.get('scaler')
    apply_pca             = mapping_data.get('apply_pca', False)

    breed    = img_data.get('breed', 'Unknown')
    img_arr  = img_data['array']
    orig_sz  = img_data.get('original_size', (224, 224))
    img_path = img_data.get('path', '')

    print(f"\n{'='*60}")
    print(f"ONLINE PHASE — Image {img_idx}: {breed}")
    print(f"{'='*60}")

    # STEP 1 — Compute logit-MAD sensitivity profile for this image.
    # Mirrors the offline Step 4 exactly: run accurate + approximate
    # forward passes, compute mean |logit_acc - logit_approx| per mult.
    # No IG, no hooks — just P+1 forward passes total.
    approx_configs_online = mapping_data.get('approx_configs', [])
    sensitivity_scaler_on = mapping_data.get('sensitivity_scaler', None)

    print(f"  Computing logit-MAD profile ({len(approx_configs_online)} multipliers)...")

    def run_forward_online(arr, axx_mult=None):
        img_t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float()
        if axx_mult is None:
            with torch.no_grad():
                out = model(img_t)
            return out.squeeze(0).cpu().numpy()
        axx_kernel    = load_approximate_multiplier_kernel(axx_mult)
        orig_forwards = {}
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                orig_forwards[name] = module.forward
                W_ = module.weight.data
                b_ = module.bias.data if module.bias is not None else None
                def _make(W__, b__, kern__):
                    def _fwd(x):
                        o = approx_matmul_submatrix_pt(x, W__.t(), kern__)
                        return o + b__ if b__ is not None else o
                    return _fwd
                module.forward = _make(W_, b_, axx_kernel)
        with torch.no_grad():
            try:
                out = model(img_t).squeeze(0).cpu().numpy()
            except Exception:
                out = np.zeros(37, dtype=np.float32)
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear) and name in orig_forwards:
                module.forward = orig_forwards[name]
        return np.where(np.isfinite(out), out, 0.0).astype(np.float32)

    # Accurate logits (one pass)
    acc_logit_on = run_forward_online(img_arr, axx_mult=None)

    # MAD per multiplier
    mad_profile = []
    for cfg in approx_configs_online:
        mult_on   = MAC_CONFIGURATIONS[cfg]['multiplier']
        apx_logit = run_forward_online(img_arr, axx_mult=mult_on)
        mad       = float(np.mean(np.abs(acc_logit_on - apx_logit)))
        mad_profile.append(mad)

    profile_arr = np.array(mad_profile, dtype=np.float32).reshape(1, -1)

    if sensitivity_scaler_on is not None:
        profile_scaled = sensitivity_scaler_on.transform(profile_arr)[0]
    else:
        profile_scaled = profile_arr[0]

    print(f"  Logit-MAD profile: {np.round(mad_profile, 4)}")

    # STEP 2 — Find nearest cluster by sensitivity distance
    dists           = np.linalg.norm(centroids - profile_scaled, axis=1)
    nearest_cluster = int(np.argmin(dists))
    print(f"  Distances to centroids: {np.round(dists, 4)}")
    print(f"  → Nearest cluster: {nearest_cluster}")

    # STEP 5 — Select corresponding multiplier from saved mapping
    selected_config = cluster_to_multiplier.get(
        nearest_cluster, list(MAC_CONFIGURATIONS.keys())[1])
    selected_mult   = MAC_CONFIGURATIONS[selected_config]['multiplier']
    print(f"  → Multiplier: {selected_config}")
    print(f"  → TPU Core:   Core {nearest_cluster + 1}")

    # STEP 6 — Compute gradient map for the new image
    img_tensor  = torch.from_numpy(img_arr).float().to(device)
    baseline    = torch.zeros_like(img_tensor)

    print(f"  Computing gradient map...")
    approx_gmap = compute_integrated_gradients(
        model, img_tensor, baseline, steps,
        axx_mult=selected_mult, device=device)

    # Save gradient maps
    tag      = breed.replace(' ', '_')
    gray_map = create_high_res_red_gradient_map(approx_gmap, orig_sz, apply_red=False)
    red_map  = create_high_res_red_gradient_map(approx_gmap, orig_sz, apply_red=True)
    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_approx_gray.png'), gray_map)
    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_approx_red.png'),  red_map)

    if os.path.exists(str(img_path)):
        Image.open(img_path).convert('RGB').save(
            os.path.join(output_dir,
                         f'online_img{img_idx}_{tag}_original.png'))

    print(f"  ✓ Gradient maps saved")

    return {
        'breed':           breed,
        'nearest_cluster': nearest_cluster,
        'selected_config': selected_config,
        'approx_gmap':     approx_gmap,
    }


# =============================================================================
# MAIN
# =============================================================================
if __name__ == '__main__':

    # ------------------------------------------------------------------
    # SETUP — copy C++ kernels (unchanged)
    # ------------------------------------------------------------------
    src_dir = '/content/drive/MyDrive/adapt-main/adapt/cpu-kernels'
    dst_dir = '/content/adapt/cpu-kernels'
    os.makedirs(dst_dir, exist_ok=True)
    os.makedirs(f'{dst_dir}/axx_mults', exist_ok=True)

    print("\n" + "="*70)
    print("DESIGN-TIME RL FRAMEWORK FOR TPU SYSTOLIC ARRAY")
    print("Homogeneous Multiplier Assignment")
    print("="*70)

    print("\n📋 Copying C++ kernel files...")
    try:
        shutil.copy(f'{src_dir}/axx_linear.cpp', f'{dst_dir}/axx_linear.cpp')
    except FileNotFoundError as e:
        print(f"  ✗ {e}")

    missing = []
    for cfg in MAC_CONFIGURATIONS.values():
        m   = cfg['multiplier']
        src = f'{src_dir}/axx_mults/{m}.h'
        dst = f'{dst_dir}/axx_mults/{m}.h'
        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            missing.append(m)

    if missing:
        print(f"  ⚠ Missing multipliers: {missing}")
        for k in list(MAC_CONFIGURATIONS.keys()):
            if MAC_CONFIGURATIONS[k]['multiplier'] in missing:
                del MAC_CONFIGURATIONS[k]
    print(f"  ✓ Available multipliers: {len(MAC_CONFIGURATIONS)}")

    # ------------------------------------------------------------------
    # STEP 1 — Detect TPU cores, assign one multiplier per core
    # ------------------------------------------------------------------
    # P is set manually to match the actual hardware.
    # On a real TPU you would query: import torch_xla.core.xla_model as xm
    #                                P = xm.xrt_world_size()
    # Here we set P = 5 to match K-Means clusters (one core per cluster).
    # Multipliers are selected from the approximate list ordered by
    # approximation level — spreading evenly from least to most aggressive.
    P = 5   # ← set this to match your actual TPU core count

    # Target range: KVA (level 3) to L2L (level 8) inclusive.
    # Levels 1-2 excluded: negligible energy saving over accurate.
    # Levels 9-10 excluded: too aggressive, quality collapses.
    TARGET_MIN_LEVEL = 3   # KVA
    TARGET_MAX_LEVEL = 8   # L2L

    all_approx = sorted(
        [c for c in MAC_CONFIGURATIONS
         if c != 'mul8s_1KV6_add16se_2TN'
         and TARGET_MIN_LEVEL <= MAC_CONFIGURATIONS[c]['level'] <= TARGET_MAX_LEVEL],
        key=lambda c: MAC_CONFIGURATIONS[c]['level']
    )

    # Select P multipliers evenly spaced across the target range
    if P >= len(all_approx):
        approx_configs = all_approx
    else:
        indices_select = np.linspace(0, len(all_approx) - 1, P, dtype=int)
        approx_configs = [all_approx[i] for i in indices_select]

    print(f"\n🔧 OFFLINE — STEP 1: Detect TPU Cores")
    print(f"  P = {P} cores (set manually to match hardware)")
    print(f"  {len(approx_configs)} multipliers selected "
          f"(evenly spaced across approximation levels)")
    print(f"  {'Core':<8} {'Multiplier':<35} {'Level'}")
    print("  " + "-"*55)
    for i, cfg in enumerate(approx_configs):
        lvl = MAC_CONFIGURATIONS[cfg]['level']
        print(f"  Core {i+1:<4} {cfg:<35} Level {lvl}")

    # ------------------------------------------------------------------
    # Load VGG16 model (unchanged)
    # ------------------------------------------------------------------
    device = torch.device('cpu')
    checkpoint_path = '/content/drive/MyDrive/Vgg_model/vgg16_oxford_pets_best.pth'

    print(f"\n📦 Loading VGG16 (37 classes)...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model = VGG16Classifier(num_classes=37)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f"  ✓ Loaded | Epoch: {checkpoint.get('epoch','N/A')} "
          f"| Acc: {checkpoint.get('test_acc', 0):.2f}%")

    # Profile layers for energy model
    profiler          = LayerProfiler(model, input_shape=(1, 3, 224, 224))
    layer_descriptors = profiler.profile()
    print(f"  ✓ {len(layer_descriptors)} layers profiled")

    # Energy normalisation
    all_energy_mJ = {}
    for cfg_name in MAC_CONFIGURATIONS:
        em = RoHNASEnergyModel(cfg_name)
        all_energy_mJ[cfg_name] = em.calculate_ig_energy(
            5, layer_descriptors)['ig_energy_mJ']
    # E_max and E_min scoped to the approx range (KVA–L2L) only.
    # energy_eff_map[c] = 0.0 for least savings (KVA), 1.0 for most (L2L).
    E_max = max(all_energy_mJ[c] for c in approx_configs)
    E_min = min(all_energy_mJ[c] for c in approx_configs)
    energy_eff_map = {
        c: (E_max - all_energy_mJ[c]) / (E_max - E_min) if E_max > E_min else 0.0
        for c in approx_configs
    }
    print(f"  ✓ E_max (approx range) = {E_max:.4f} mJ")
    print(f"  ✓ E_min (approx range) = {E_min:.4f} mJ")

    # ------------------------------------------------------------------
    # STEP 2 — Collect training images from dataset
    # ------------------------------------------------------------------
    print(f"\n📸 OFFLINE — STEP 2: Collect Training Images")
    data_root = '/content/datasets'
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    dataset = datasets.OxfordIIITPet(
        root=data_root, split='trainval', download=True, transform=transform)

    np.random.seed(2024)
    indices     = np.random.choice(len(dataset), 100, replace=False)
    images_data = []
    for idx in indices:
        img_tensor, label = dataset[idx]
        img_path  = dataset._images[idx]
        img_pil   = Image.open(img_path).convert('RGB')
        img_np    = img_tensor.permute(1, 2, 0).numpy()
        breed     = dataset.classes[label].replace('_', ' ').title()
        images_data.append({
            'array':         img_np,
            'path':          img_path,
            'original_size': img_pil.size,
            'breed':         breed,
            'label':         label
        })
    print(f"  ✓ {len(images_data)} training images collected")
    breed_counts = {}
    for d in images_data:
        breed_counts[d['breed']] = breed_counts.get(d['breed'], 0) + 1
    print(f"  Breeds ({len(breed_counts)} unique): {sorted(breed_counts.keys())}")

    # ------------------------------------------------------------------
    # OFFLINE PHASE (Steps 3–10)
    # ------------------------------------------------------------------
    offline_output = '/content/tpu_rl_offline'
    mapping_data   = offline_phase(
        model             = model,
        images_data       = images_data,
        device            = device,
        layer_descriptors = layer_descriptors,
        all_energy_mJ     = all_energy_mJ,
        E_max             = E_max,
        energy_eff_map    = energy_eff_map,
        output_dir        = offline_output,
        approx_configs    = approx_configs,
        steps             = 5
    )

    # ------------------------------------------------------------------
    # ONLINE PHASE — 3 new unseen images
    # ------------------------------------------------------------------
    print("\n\n" + "="*70)
    print("ONLINE PHASE — RUNTIME ROUTING")
    print("No RL runs at runtime. Cluster lookup only.")
    print("="*70)

    online_output = '/content/tpu_rl_online'
    np.random.seed(999)
    online_indices = np.random.choice(len(dataset), 3, replace=False)

    online_results = []
    for i, oidx in enumerate(online_indices):
        img_tensor, label = dataset[oidx]
        img_path  = dataset._images[oidx]
        img_pil   = Image.open(img_path).convert('RGB')
        img_np    = img_tensor.permute(1, 2, 0).numpy()
        breed     = dataset.classes[label].replace('_', ' ').title()
        new_img   = {
            'array':         img_np,
            'path':          img_path,
            'original_size': img_pil.size,
            'breed':         breed
        }
        result = online_phase(
            model        = model,
            img_data     = new_img,
            mapping_data = mapping_data,
            device       = device,
            output_dir   = online_output,
            steps        = 5,
            img_idx      = i
        )
        online_results.append(result)

    # ------------------------------------------------------------------
    # FINAL SUMMARY
    # ------------------------------------------------------------------
    print("\n\n" + "="*70)
    print("FINAL TPU CORE ASSIGNMENT TABLE")
    print("="*70)
    print(f"  {'Cluster':<10} {'Multiplier':<35} {'Core':<8} {'Level'}")
    print("  " + "-"*60)
    for k, cfg in mapping_data['cluster_to_multiplier'].items():
        lvl = MAC_CONFIGURATIONS[cfg]['level']
        print(f"  {k:<10} {cfg:<35} Core {k+1:<4} Level {lvl}")

    print(f"\n  Online routing results:")
    print(f"  {'Breed':<25} {'Cluster':<10} {'Multiplier'}")
    print("  " + "-"*60)
    for r in online_results:
        print(f"  {r['breed']:<25} {r['nearest_cluster']:<10} {r['selected_config']}")

    print(f"\n✅ COMPLETE")
    print(f"  Offline outputs → {offline_output}")
    print(f"  Online outputs  → {online_output}")
    print("="*70)


DESIGN-TIME RL FRAMEWORK FOR TPU SYSTOLIC ARRAY
Homogeneous Multiplier Assignment

📋 Copying C++ kernel files...
  ✓ Available multipliers: 11

🔧 OFFLINE — STEP 1: Detect TPU Cores
  P = 5 cores (set manually to match hardware)
  5 multipliers selected (evenly spaced across approximation levels)
  Core     Multiplier                          Level
  -------------------------------------------------------
  Core 1    mul8s_1KVA_add16se_2TN              Level 3
  Core 2    mul8s_1KVP_add16se_2TN              Level 4
  Core 3    mul8s_1KVQ_add16se_2TN              Level 5
  Core 4    mul8s_1KX5_add16se_2TN              Level 6
  Core 5    mul8s_1L2L_add16se_2TN              Level 8

📦 Loading VGG16 (37 classes)...
  ✓ Loaded | Epoch: 30 | Acc: 87.93%
  ✓ 16 layers profiled
  ✓ E_max (approx range) = 0.9497 mJ
  ✓ E_min (approx range) = 0.6927 mJ

📸 OFFLINE — STEP 2: Collect Training Images
  ✓ 100 training images collected
  Breeds (36 unique): ['Abyssinian', 'American Bulldog', 'America